In [ ]:
!pip install labelme opencv-python pillow scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 17.8 MB/s eta 0:00:0000:0100:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.9/59.9 MB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 276.4/276.4 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 6.2 MB/s eta 0:00:00
  Created wheel for labelme: filename=labelme-5.6.1-py3-none-any.whl size=1439272 sha256=9

In [ ]:
%pip install ultralytics
import ultralytics
ultralytics.checks()

Ultralytics 8.3.71 🚀 Python-3.11.11 torch-2.5.1+cu124 CPU (Intel Xeon 2.20GHz)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 33.3/107.7 GB disk)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
DIRECT TO A FODLER OR ELSE YOU WILL HAVE A HARD TIME, keep in a folder

In [ ]:
!unzip /content/drive/MyDrive/Annotations/Collab_Test_Dataset.zip -d /content/drive/MyDrive/Annotations

Archive:  /content/drive/MyDrive/Annotations/Collab_Test_Dataset.zip
replace /content/drive/MyDrive/Annotations/frame_0079_jpg.rf.a0cebbd9a827814a9b8f8606c25bec63.json? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [ ]:
import os
import shutil
import math
import json
import cv2
import PIL.Image
from collections import OrderedDict
from sklearn.model_selection import train_test_split
from labelme import utils

class Labelme2YOLO(object):
    """
    A class to convert Labelme annotations (JSON) to YOLO annotation format.
    Also optionally creates segmentation format (for YOLOv5 segmentations).
    """

    def __init__(self, json_dir, to_seg=True):
        """
        Parameters:
            json_dir (str): Path to the folder containing Labelme JSON files.
            to_seg (bool): If True, convert to YOLOv5 segmentation format (v7.0 style).
        """
        self._json_dir = '/content/drive/MyDrive/Annotations'
        self._label_id_map = self._get_label_id_map(self._json_dir)
        self._to_seg = to_seg

        # Instead of creating "YOLODataset" subfolder, save files directly in json_dir
        self._save_path_pfx = self._json_dir

    def _make_train_val_dir(self):
        """
        Creates the 'images/train', 'images/val', 'labels/train', 'labels/val' directories
        in self._json_dir. If they exist, they are removed first.
        """
        self._label_dir_path = os.path.join(self._save_path_pfx, 'labels')
        self._image_dir_path = os.path.join(self._save_path_pfx, 'images')

        for p in [
            os.path.join(self._label_dir_path, 'train'),
            os.path.join(self._label_dir_path, 'val'),
            os.path.join(self._image_dir_path, 'train'),
            os.path.join(self._image_dir_path, 'val')
        ]:
            if os.path.exists(p):
                shutil.rmtree(p)
            os.makedirs(p)

    def _get_label_id_map(self, json_dir):
        """
        Scans all JSON files in the directory and collects unique labels,
        then returns an OrderedDict that maps label -> label_id.
        """
        label_set = set()
        for file_name in os.listdir(json_dir):
            if file_name.endswith('.json'):
                json_path = os.path.join(json_dir, file_name)
                with open(json_path, 'r') as f:
                    data = json.load(f)
                for shape in data.get('shapes', []):
                    label_set.add(shape['label'])
        return OrderedDict([(label, label_id) for label_id, label in enumerate(label_set)])

    def _train_test_split(self, folders, json_names, val_size):
        """
        Splits the JSON files into train/val subsets.
        If 'train/' or 'val/' directories already exist in the JSON directory,
        it attempts to read from them instead of random splitting.
        """
        # If user manually split into train/val folders, read from them
        if len(folders) > 0 and 'train' in folders and 'val' in folders:
            train_folder = os.path.join(self._json_dir, 'train/')
            val_folder = os.path.join(self._json_dir, 'val/')

            train_json_names = [
                fname for fname in os.listdir(train_folder) if fname.endswith('.json')
            ]
            val_json_names = [
                fname for fname in os.listdir(val_folder) if fname.endswith('.json')
            ]
            return train_json_names, val_json_names

        # Otherwise do a random split
        train_idxs, val_idxs = train_test_split(range(len(json_names)), test_size=val_size)
        train_json_names = [json_names[i] for i in train_idxs]
        val_json_names = [json_names[i] for i in val_idxs]
        return train_json_names, val_json_names

    def convert(self, val_size=0.1):
        """
        Converts all JSONs in the directory to YOLO format, performing a train/val split.

        Parameters:
            val_size (float): Percentage (0-1) that goes into validation set.
        """
        json_names = [
            file_name for file_name in os.listdir(self._json_dir)
            if os.path.isfile(os.path.join(self._json_dir, file_name)) and file_name.endswith('.json')
        ]
        folders = [
            file_name for file_name in os.listdir(self._json_dir)
            if os.path.isdir(os.path.join(self._json_dir, file_name))
        ]

        train_json_names, val_json_names = self._train_test_split(folders, json_names, val_size)

        self._make_train_val_dir()

        # Convert labelme object to yolo format and save them
        for target_dir, subset_json_names in zip(('train', 'val'), (train_json_names, val_json_names)):
            for json_name in subset_json_names:
                json_path = os.path.join(self._json_dir, json_name)
                with open(json_path, 'r') as f:
                    json_data = json.load(f)
                print(f'Converting {json_name} for {target_dir} ...')

                # Save image to YOLO dataset folder
                img_path = self._save_yolo_image(json_data, json_name, self._image_dir_path, target_dir)

                # Parse shapes and convert to YOLO
                yolo_obj_list = self._get_yolo_object_list(json_data, img_path)

                # Save as .txt for YOLO
                self._save_yolo_label(json_name, self._label_dir_path, target_dir, yolo_obj_list)

        print('Generating dataset.yaml file ...')
        self._save_dataset_yaml()

    def convert_one(self, json_name):
        """
        Converts a single JSON file to YOLO format (no train/val split).
        """
        json_path = os.path.join(self._json_dir, json_name)
        with open(json_path, 'r') as f:
            json_data = json.load(f)

        print(f'Converting {json_name} ...')

        # Save image in the same directory (no train/val subfolders).
        img_path = self._save_yolo_image(json_data, json_name, self._json_dir, '')

        yolo_obj_list = self._get_yolo_object_list(json_data, img_path)
        self._save_yolo_label(json_name, self._json_dir, '', yolo_obj_list)

    def _get_yolo_object_list(self, json_data, img_path):
        """
        Parse shapes from a single JSON's annotation data and convert them to YOLO lists.
        Returns a list of lines to be saved in the corresponding .txt file.
        """
        img = cv2.imread(img_path)
        if img is None:
            raise ValueError(f"Could not read image at {img_path}")
        img_h, img_w, _ = img.shape

        yolo_obj_list = []
        for shape in json_data.get('shapes', []):
            shape_type = shape.get('shape_type', 'polygon')
            if shape_type == 'circle':
                yolo_obj = self._get_circle_shape_yolo_object(shape, img_h, img_w)
            else:
                yolo_obj = self._get_other_shape_yolo_object(shape, img_h, img_w)
            yolo_obj_list.append(yolo_obj)
        return yolo_obj_list

    def _get_circle_shape_yolo_object(self, shape, img_h, img_w):
        """
        Convert circle (center + radius) to YOLO bounding box or segmentation.
        """
        label_id = self._label_id_map[shape['label']]
        obj_center_x, obj_center_y = shape['points'][0]

        radius = math.sqrt(
            (obj_center_x - shape['points'][1][0]) ** 2 +
            (obj_center_y - shape['points'][1][1]) ** 2
        )

        # For segmentation, approximate the circle with polygon points
        if self._to_seg:
            # Start with [class_id], then the polygon points
            retval = [label_id]

            # Increase 'n_part' if you want a smoother circle approximation
            n_part = max(int(radius / 10), 4)
            n_part2 = n_part * 2

            # We'll create 4 quadrants of partial arcs
            pt_quad = [[] for _ in range(4)]

            # 1st quadrant points
            pt_quad[0] = [
                [
                    obj_center_x + math.cos(i * math.pi / n_part2) * radius,
                    obj_center_y - math.sin(i * math.pi / n_part2) * radius
                ]
                for i in range(1, n_part)
            ]
            # 2nd quadrant (mirror in x)
            pt_quad[1] = [[(obj_center_x * 2 - x), y] for x, y in pt_quad[0]]
            pt_quad[1].reverse()

            # 4th quadrant (mirror in y)
            pt_quad[3] = [[x, (obj_center_y * 2 - y)] for x, y in pt_quad[0]]
            pt_quad[3].reverse()

            # 3rd quadrant (mirror in x from 4th)
            pt_quad[2] = [[(obj_center_x * 2 - x), y] for x, y in pt_quad[3]]
            pt_quad[2].reverse()

            # Add edges to close the circle:
            pt_quad[0].append([obj_center_x, obj_center_y - radius]) # top
            pt_quad[1].append([obj_center_x - radius, obj_center_y]) # left
            pt_quad[2].append([obj_center_x, obj_center_y + radius]) # bottom
            pt_quad[3].append([obj_center_x + radius, obj_center_y]) # right

            # Convert to normalized [x, y]
            for quadrant in pt_quad:
                for x, y in quadrant:
                    nx = round(float(x) / img_w, 6)
                    ny = round(float(y) / img_h, 6)
                    retval.extend([nx, ny])

            return retval

        else:
            # Standard bounding box
            obj_w = 2 * radius
            obj_h = 2 * radius

            yolo_center_x= round(float(obj_center_x / img_w), 6)
            yolo_center_y= round(float(obj_center_y / img_h), 6)
            yolo_w = round(float(obj_w / img_w), 6)
            yolo_h = round(float(obj_h / img_h), 6)

            return (label_id, yolo_center_x, yolo_center_y, yolo_w, yolo_h)

    def _get_other_shape_yolo_object(self, shape, img_h, img_w):
        """
        Convert polygons, rectangles, lines, etc. to YOLO bounding box or segmentation format.
        """
        label_id = self._label_id_map[shape['label']]

        if self._to_seg:
            # For segmentation, we store [label_id, x1, y1, x2, y2, ...]
            retval = [label_id]
            for pt in shape['points']:
                nx = round(float(pt[0]) / img_w, 6)
                ny = round(float(pt[1]) / img_h, 6)
                retval.extend([nx, ny])
            return retval
        else:
            x_coords = [p[0] for p in shape['points']]
            y_coords = [p[1] for p in shape['points']]

            x_min, x_max = min(x_coords), max(x_coords)
            y_min, y_max = min(y_coords), max(y_coords)

            obj_w = x_max - x_min
            obj_h = y_max - y_min

            yolo_center_x = round((x_min + obj_w / 2) / img_w, 6)
            yolo_center_y = round((y_min + obj_h / 2) / img_h, 6)
            yolo_w = round(obj_w / img_w, 6)
            yolo_h = round(obj_h / img_h, 6)

            return (label_id, yolo_center_x, yolo_center_y, yolo_w, yolo_h)

    def _save_yolo_label(self, json_name, label_dir_path, target_dir, yolo_obj_list):
        """
        Given the YOLO objects, writes a .txt file with each line: 'class cx cy w h'
        (or segmentation format if self._to_seg) into the correct folder.
        """
        # If target_dir is empty, save txt in the root of label_dir_path
        txt_folder = os.path.join(label_dir_path, target_dir)
        if not os.path.exists(txt_folder):
            os.makedirs(txt_folder, exist_ok=True)
        txt_path = os.path.join(txt_folder, json_name.replace('.json', '.txt'))

        with open(txt_path, 'w') as f:
            for idx, yolo_obj in enumerate(yolo_obj_list):
                line_str = " ".join(str(x) for x in yolo_obj)
                # Add newline except for the last object
                if idx < len(yolo_obj_list) - 1:
                    line_str += "\n"
                f.write(line_str)

    def _save_yolo_image(self, json_data, json_name, image_dir_path, target_dir):
        """
        Convert the base64-encoded image in JSON to an actual .png image and save it.
        """
        img_name = json_name.replace('.json', '.png')
        dst_folder = os.path.join(image_dir_path, target_dir)
        if not os.path.exists(dst_folder):
            os.makedirs(dst_folder, exist_ok=True)
        dst_path = os.path.join(dst_folder, img_name)

        # Only create the image if it doesn't already exist
        if not os.path.exists(dst_path):
            img = utils.img_b64_to_arr(json_data['imageData'])
            PIL.Image.fromarray(img).save(dst_path)

        return dst_path

    def _save_dataset_yaml(self):
        """
        Creates a 'dataset.yaml' file containing paths to train/val sets,
        the number of classes, and the class names.
        """
        yaml_path = os.path.join(self._save_path_pfx, 'dataset.yaml')
        train_images_path = os.path.join(self._image_dir_path, 'train')
        val_images_path   = os.path.join(self._image_dir_path, 'val')

        with open(yaml_path, 'w') as yf:
            yf.write(f"train: {train_images_path}\n")
            yf.write(f"val: {val_images_path}\n\n")
            yf.write(f"nc: {len(self._label_id_map)}\n\n")

            names_str = ", ".join(f"'{label}'" for label in self._label_id_map.keys())
            yf.write(f"names: [{names_str}]\n")

# --------- Convenience functions ----------

def convert_all_jsons(json_dir, val_size=0.1, to_seg=True):
    """
    Instantiate the class and convert all JSONs in `json_dir`
    with a train/val split. Output is saved directly to `json_dir`.
    """
    converter = Labelme2YOLO(json_dir=json_dir, to_seg=to_seg)
    converter.convert(val_size=val_size)

def convert_single_json(json_dir, json_name, to_seg=False):
    """
    Convert a single JSON file in `json_dir`.
    Output (label txt + .png) is saved directly to `json_dir`.
    """
    converter = Labelme2YOLO(json_dir=json_dir, to_seg=to_seg)
    converter.convert_one(json_name)


In [ ]:
# Set the path to your annotation folder (update this path as needed)
json_dir = "/content/Annotations"  # or "/content/drive/MyDrive/your_folder/Annotations"
to_seg = True     # set to False if you don't need segmentation format
val_size = 0.1    # validation set size

# Create an instance of the converter and run conversion
converter = Labelme2YOLO(json_dir, to_seg=to_seg)
converter.convert(val_size=val_size)

Converting frame_0079_jpg.rf.bdee8bcb391388a2a0079f84ba2ede87.json for train ...
Converting frame_0080_jpg.rf.5f39cf7afc4850c53a78c07a84351b01.json for train ...
Converting frame_0080_jpg.rf.ae53bcbdee7a578bdf395b143cc9f8a3.json for train ...
Converting frame_0079_jpg.rf.fde94aff10c531cdc890c230c7bd40ce.json for train ...
Converting frame_0079_jpg.rf.e7d5fc277a97df5450355dd7d6c9ff0b.json for train ...
Converting frame_0080_jpg.rf.8a0f490378ae1ae0205e6296d3796cd7.json for train ...
Converting frame_0080_jpg.rf.1630ae03ae2a02fcc87c7a689e30cebf.json for train ...
Converting frame_0080_jpg.rf.b42e58464b75a60ef2648b8b4cfe7ec7.json for train ...
Converting frame_0079_jpg.rf.db8b6f15af7ced251e1a58fbe0388a93.json for train ...
Converting frame_0080_jpg.rf.6cf87f438c8cf15f315d3c8d5c332d34.json for train ...
Converting frame_0080_jpg.rf.a433755fb0cd2b6a05c255b5589ecfc8.json for train ...
Converting frame_0080_jpg.rf.6ca59af47c6c55a05ef9a4169743020c.json for train ...
Converting frame_0080_jpg.rf

In [ ]:
# Train YOLO11n on COCO8 for 3 epochs
!yolo task=segment mode=train model=yolo11n-seg.pt data=/content/drive/MyDrive/Annotations/dataset.yaml epochs=3 imgsz=640

Ultralytics 8.3.71 🚀 Python-3.11.11 torch-2.5.1+cu124 CPU (Intel Xeon 2.20GHz)
engine/trainer: task=segment, mode=train, model=yolo11n-seg.pt, data=/content/drive/MyDrive/Annotations/dataset.yaml, epochs=10, time=None, patience=100, batch=16, imgsz=640, save=True, save_period=-1, cache=False, device=None, workers=8, project=None, name=train2, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False, save_frames=False, save_txt=False, save_conf=False, save_crop=False, show_labels=True, sh

In [ ]:
#Change model= to best.pt when the your model is ready
!yolo export model=yolo11n.pt format=tflite